# sat_beyond_health — Satélite Beyond Health

Este notebook construye el **satélite de Beyond Health**, que es la tabla central donde
se consolida toda la información de los asegurados del sistema Beyond Health.

## ¿Qué hace este notebook?

Toma información de **6 tablas fuente** del sistema Beyond Health y las une en una sola tabla
que contiene los datos completos de cada asegurado, identificando claramente si es
**titular** (el asegurado principal del contrato) o **beneficiario** (familiar u otra
persona incluida en el contrato).

## Tablas fuente

| Tabla | Qué contiene |
|---|---|
| `bh_sa_member` | Lista de todos los miembros (titulares y beneficiarios) de cada contrato |
| `bh_sa_affiliation_contract` | Información del contrato de afiliación |
| `bh_sa_person` | Datos personales de cada persona (nombre, documento, etc.) |
| `bh_sa_institution` | Datos de la institución o empresa que suscribe el contrato |
| `bh_sa_address` | Direcciones registradas de cada persona |
| `bh_sa_city` | Catálogo de ciudades con su código y nombre |

## Tabla destino

`uc_axa_cli.silver.sat_beyond_health`

## Regla de negocio clave

- **Titular**: la persona se obtiene desde el **contrato** (`aco.per_ncode`)
- **Beneficiario**: la persona se obtiene desde el **member** (`mem.per_ncode`)

## Paso 1 — Preparar la dirección residencial de cada persona

Antes de armar el satélite, necesitamos tener lista la **dirección residencial** de cada
persona. El problema es que una persona puede tener varias direcciones registradas
(residencial, laboral, de correo, etc.), por eso hacemos este paso previo.

**¿Qué hacemos aquí?**

1. De la tabla de direcciones (`bh_sa_address`), filtramos solo las que son de tipo
   **residencial** (`lty_ncode = 1`).
2. Cruzamos con el catálogo de ciudades para traer el **código de ciudad**.
3. Agrupamos por persona para quedarnos con **una sola fila por persona**
   (usamos `MAX` para tomar un valor cuando hay más de uno).

El resultado es una tabla auxiliar `residencial` con una fila por persona que tiene:
su dirección residencial y el código de su ciudad.

## Paso 2 — Preparar el nombre de la ciudad

Con el código de ciudad que obtuvimos en el paso anterior, buscamos el
**nombre legible de la ciudad** en el catálogo `bh_sa_city`.

Esto nos permite mostrar, por ejemplo, `"BOGOTA D.C."` en lugar de solo el código `"11001"`.

## Paso 3 — Construir los registros de TITULARES

Aquí construimos la información completa de los **titulares** del contrato.

**¿Cómo identificamos al titular?**  
En la tabla `bh_sa_member`, el titular es el miembro cuyo `per_ncode` coincide con el
`per_ncode` del contrato de afiliación (`mem.per_ncode = aco.per_ncode`). Esto garantiza
**exactamente un registro TITULAR por contrato**, sin importar cuántos beneficiarios tenga.

Los datos personales (nombre, documento, dirección) se obtienen desde el **contrato**
(`aco.per_ncode`), porque el titular es quien firma el contrato.

**Joins que se hacen:**

| Unión | Para qué |
|---|---|
| `member` + `contrato` | Traer los datos del contrato al que pertenece cada miembro |
| `contrato` + `persona` | Traer los datos personales del **titular del contrato** |
| `contrato` + `institución` | Traer los datos de la empresa u organización del contrato |
| `contrato` + `residencial` | Traer la dirección del **titular** (via `aco.per_ncode`) |
| `residencial` + `ciudad` | Traer el nombre de la ciudad del titular |

**Filtro clave:** `WHERE mem.per_ncode = aco.per_ncode`

Al final, cada fila queda marcada con `rol = 'TITULAR'`.

**Nota sobre los nombres de columnas:**  
Cuando una misma columna existe en varias tablas (por ejemplo `fec_cargue` existe en
member, contrato y persona), se le agrega un prefijo para saber de dónde viene:
`mem_fec_cargue`, `aco_fec_cargue`, `per_fec_cargue`.

## Paso 4 — Construir los registros de BENEFICIARIOS

Aquí construimos la información completa de los **beneficiarios** del contrato.

**¿En qué se diferencia del titular?**  
Hay **tres diferencias** respecto al bloque anterior:

1. **Filtro de rol:** Solo se incluyen miembros cuyo `per_ncode` es **distinto** al del
   contrato (`WHERE mem.per_ncode <> aco.per_ncode`). Contratos sin beneficiarios no
   producen ninguna fila en este CTE.
2. La persona del beneficiario viene del **member** (`mem.per_ncode`), no del contrato.
3. La dirección residencial también se busca con el `per_ncode` del **member**.

Un contrato puede tener **0, 1 o N beneficiarios** — cada beneficiario genera su propia
fila con su documento y datos personales únicos.

Todo lo demás (columnas, estructura, prefijos) es idéntico al bloque de titulares.  
Al final, cada fila queda marcada con `rol = 'BENEFICIARIO'`.

## Paso 5 — Unir titulares y beneficiarios en una sola tabla

Con `UNION ALL` juntamos los registros de titulares y beneficiarios en una sola tabla.

La columna `rol` nos permite distinguir en todo momento quién es quién:
- `TITULAR` → persona principal del contrato (un registro por contrato)
- `BENEFICIARIO` → familiar u otro miembro incluido en el contrato (0 a N registros por contrato)

**Resultado esperado:** El total de filas es igual al total de miembros en `bh_sa_member`,
ya que los filtros `mem.per_ncode = aco.per_ncode` y `mem.per_ncode <> aco.per_ncode`
son mutuamente excluyentes y exhaustivos — cada fila de member cae exactamente en uno de los dos grupos.

## Ejecución — Crear el satélite Beyond Health

La siguiente celda ejecuta todo el proceso descrito arriba en un solo paso.
Crea (o reemplaza) la tabla `uc_axa_cli.silver.sat_beyond_health` con todos los datos.

> ⚠️ **Importante:** Este proceso borra y recrea la tabla completa cada vez que se ejecuta.
> Esto garantiza que siempre refleje el estado más reciente de las tablas fuente.

In [ ]:
%sql

CREATE OR REPLACE TABLE axa_col_slv_dv.stg_cliente.sat_beyond_health
USING DELTA AS

-- ============================================================
-- PASO 1: Dirección residencial por persona
-- ============================================================
WITH residencial AS (
    SELECT
        a.per_ncode             AS res_per_ncode,
        MAX(a.add_caddress)     AS dir_res,
        MAX(c.cit_clegalcode)   AS ciu_res_codigo
    FROM axa_col_slv_dv.core_bh.bh_sa_address a
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_city c
        ON a.cit_ncode = c.cit_ncode
    WHERE a.lty_ncode = 1
    GROUP BY a.per_ncode
),

-- ============================================================
-- PASO 2: Ciudad y país
-- bh_sa_city tiene dep_ncode para ciudades colombianas.
-- Si dep_ncode IS NOT NULL → Colombia; si IS NULL → Exterior.
-- pais queda NULL cuando no hay ciudad registrada (LEFT JOIN).
-- ============================================================
ciudad AS (
    SELECT
        c.cit_clegalcode        AS ciu_res_codigo,
        c.cit_cname             AS ciudad_residencia,
        CASE
            WHEN c.dep_ncode IS NOT NULL THEN 'Colombia'
            ELSE 'Exterior'
        END                     AS pais
    FROM axa_col_slv_dv.core_bh.bh_sa_city c
),

-- ============================================================
-- PASO 3: Titulares (mem.per_ncode = aco.per_ncode)
-- ============================================================
titular AS (
    SELECT
        mem.* EXCEPT (aco_ncode, per_ncode, ins_ncode,
                      fec_cargue, fec_actualizacion, fec_eliminacion,
                      tipo_proceso, fec_movimiento,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        mem.aco_ncode           AS mem_aco_ncode,
        mem.per_ncode           AS mem_per_ncode,
        mem.ins_ncode           AS mem_ins_ncode,
        mem.fec_cargue          AS mem_fec_cargue,
        mem.fec_actualizacion   AS mem_fec_actualizacion,
        mem.fec_eliminacion     AS mem_fec_eliminacion,
        mem.tipo_proceso        AS mem_tipo_proceso,
        mem.fec_movimiento      AS mem_fec_movimiento,
        mem.COMPANIA            AS mem_COMPANIA,
        mem.TIPO_CARGA          AS mem_TIPO_CARGA,
        mem.PERIODO             AS mem_PERIODO,
        mem.FECHA_CARGUE        AS mem_FECHA_CARGUE,
        aco.* EXCEPT (per_ncode, ins_ncode, add_ncode, MST_NCODE,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        aco.per_ncode           AS aco_per_ncode,
        aco.ins_ncode           AS aco_ins_ncode,
        aco.add_ncode           AS aco_ADD_NCODE,
        aco.MST_NCODE           AS aco_MST_NCODE,
        aco.FEC_CARGUE          AS aco_FEC_CARGUE,
        aco.FEC_ACTUALIZACION   AS aco_FEC_ACTUALIZACION,
        aco.FEC_ELIMINACION     AS aco_FEC_ELIMINACION,
        aco.TIPO_PROCESO        AS aco_TIPO_PROCESO,
        aco.FEC_MOVIMIENTO      AS aco_FEC_MOVIMIENTO,
        aco.COMPANIA            AS aco_COMPANIA,
        aco.TIPO_CARGA          AS aco_TIPO_CARGA,
        aco.PERIODO             AS aco_PERIODO,
        aco.FECHA_CARGUE        AS aco_FECHA_CARGUE,
        per.* EXCEPT (MST_NCODE, HOL_NCODE, EAC_NCODE,
                      TID_NCODE, PER_CIDENTIFICATIONNUMBER,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        per.TID_NCODE                 AS tipo_documento,
        per.PER_CIDENTIFICATIONNUMBER AS numero_documento,
        per.MST_NCODE           AS per_MST_NCODE,
        per.HOL_NCODE           AS per_HOL_NCODE,
        per.EAC_NCODE           AS actividad_economica,
        per.FEC_CARGUE          AS per_FEC_CARGUE,
        per.FEC_ACTUALIZACION   AS per_FEC_ACTUALIZACION,
        per.FEC_ELIMINACION     AS per_FEC_ELIMINACION,
        per.TIPO_PROCESO        AS per_TIPO_PROCESO,
        per.FEC_MOVIMIENTO      AS per_FEC_MOVIMIENTO,
        per.COMPANIA            AS per_COMPANIA,
        per.TIPO_CARGA          AS per_TIPO_CARGA,
        per.PERIODO             AS per_PERIODO,
        per.FECHA_CARGUE        AS per_FECHA_CARGUE,
        ins.* EXCEPT (EAC_NCODE, HOL_NCODE,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        ins.EAC_NCODE           AS ins_EAC_NCODE,
        ins.HOL_NCODE           AS ins_HOL_NCODE,
        ins.FEC_CARGUE          AS ins_FEC_CARGUE,
        ins.FEC_ACTUALIZACION   AS ins_FEC_ACTUALIZACION,
        ins.FEC_ELIMINACION     AS ins_FEC_ELIMINACION,
        ins.TIPO_PROCESO        AS ins_TIPO_PROCESO,
        ins.FEC_MOVIMIENTO      AS ins_FEC_MOVIMIENTO,
        ins.COMPANIA            AS ins_COMPANIA,
        ins.TIPO_CARGA          AS ins_TIPO_CARGA,
        ins.PERIODO             AS ins_PERIODO,
        ins.FECHA_CARGUE        AS ins_FECHA_CARGUE,
        res.dir_res,
        res.ciu_res_codigo,
        ciudad.ciudad_residencia,
        ciudad.pais,
        'TITULAR'               AS rol
    FROM axa_col_slv_dv.core_bh.bh_sa_member mem
    INNER JOIN axa_col_slv_dv.core_bh.bh_sa_affiliation_contract aco
        ON mem.aco_ncode = aco.aco_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_person per
        ON aco.per_ncode = per.per_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_institution ins
        ON aco.ins_ncode = ins.ins_ncode
    LEFT JOIN residencial res
        ON aco.per_ncode = res.res_per_ncode
    LEFT JOIN ciudad
        ON res.ciu_res_codigo = ciudad.ciu_res_codigo
    WHERE mem.per_ncode = aco.per_ncode
),

-- ============================================================
-- PASO 4: Beneficiarios (mem.per_ncode <> aco.per_ncode)
-- ============================================================
beneficiario AS (
    SELECT
        mem.* EXCEPT (aco_ncode, per_ncode, ins_ncode,
                      fec_cargue, fec_actualizacion, fec_eliminacion,
                      tipo_proceso, fec_movimiento,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        mem.aco_ncode           AS mem_aco_ncode,
        mem.per_ncode           AS mem_per_ncode,
        mem.ins_ncode           AS mem_ins_ncode,
        mem.fec_cargue          AS mem_fec_cargue,
        mem.fec_actualizacion   AS mem_fec_actualizacion,
        mem.fec_eliminacion     AS mem_fec_eliminacion,
        mem.tipo_proceso        AS mem_tipo_proceso,
        mem.fec_movimiento      AS mem_fec_movimiento,
        mem.COMPANIA            AS mem_COMPANIA,
        mem.TIPO_CARGA          AS mem_TIPO_CARGA,
        mem.PERIODO             AS mem_PERIODO,
        mem.FECHA_CARGUE        AS mem_FECHA_CARGUE,
        aco.* EXCEPT (per_ncode, ins_ncode, add_ncode, MST_NCODE,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        aco.per_ncode           AS aco_per_ncode,
        aco.ins_ncode           AS aco_ins_ncode,
        aco.add_ncode           AS aco_ADD_NCODE,
        aco.MST_NCODE           AS aco_MST_NCODE,
        aco.FEC_CARGUE          AS aco_FEC_CARGUE,
        aco.FEC_ACTUALIZACION   AS aco_FEC_ACTUALIZACION,
        aco.FEC_ELIMINACION     AS aco_FEC_ELIMINACION,
        aco.TIPO_PROCESO        AS aco_TIPO_PROCESO,
        aco.FEC_MOVIMIENTO      AS aco_FEC_MOVIMIENTO,
        aco.COMPANIA            AS aco_COMPANIA,
        aco.TIPO_CARGA          AS aco_TIPO_CARGA,
        aco.PERIODO             AS aco_PERIODO,
        aco.FECHA_CARGUE        AS aco_FECHA_CARGUE,
        per.* EXCEPT (MST_NCODE, HOL_NCODE, EAC_NCODE,
                      TID_NCODE, PER_CIDENTIFICATIONNUMBER,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        per.TID_NCODE                 AS tipo_documento,
        per.PER_CIDENTIFICATIONNUMBER AS numero_documento,
        per.MST_NCODE           AS per_MST_NCODE,
        per.HOL_NCODE           AS per_HOL_NCODE,
        per.EAC_NCODE           AS per_EAC_NCODE,
        per.FEC_CARGUE          AS per_FEC_CARGUE,
        per.FEC_ACTUALIZACION   AS per_FEC_ACTUALIZACION,
        per.FEC_ELIMINACION     AS per_FEC_ELIMINACION,
        per.TIPO_PROCESO        AS per_TIPO_PROCESO,
        per.FEC_MOVIMIENTO      AS per_FEC_MOVIMIENTO,
        per.COMPANIA            AS per_COMPANIA,
        per.TIPO_CARGA          AS per_TIPO_CARGA,
        per.PERIODO             AS per_PERIODO,
        per.FECHA_CARGUE        AS per_FECHA_CARGUE,
        ins.* EXCEPT (EAC_NCODE, HOL_NCODE,
                      FEC_CARGUE, FEC_ACTUALIZACION, FEC_ELIMINACION,
                      TIPO_PROCESO, FEC_MOVIMIENTO,
                      COMPANIA, TIPO_CARGA, PERIODO, FECHA_CARGUE),
        ins.EAC_NCODE           AS ins_EAC_NCODE,
        ins.HOL_NCODE           AS ins_HOL_NCODE,
        ins.FEC_CARGUE          AS ins_FEC_CARGUE,
        ins.FEC_ACTUALIZACION   AS ins_FEC_ACTUALIZACION,
        ins.FEC_ELIMINACION     AS ins_FEC_ELIMINACION,
        ins.TIPO_PROCESO        AS ins_TIPO_PROCESO,
        ins.FEC_MOVIMIENTO      AS ins_FEC_MOVIMIENTO,
        ins.COMPANIA            AS ins_COMPANIA,
        ins.TIPO_CARGA          AS ins_TIPO_CARGA,
        ins.PERIODO             AS ins_PERIODO,
        ins.FECHA_CARGUE        AS ins_FECHA_CARGUE,
        res.dir_res,
        res.ciu_res_codigo,
        ciudad.ciudad_residencia,
        ciudad.pais,
        'BENEFICIARIO'          AS rol
    FROM axa_col_slv_dv.core_bh.bh_sa_member mem
    INNER JOIN axa_col_slv_dv.core_bh.bh_sa_affiliation_contract aco
        ON mem.aco_ncode = aco.aco_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_person per
        ON mem.per_ncode = per.per_ncode
    LEFT JOIN axa_col_slv_dv.core_bh.bh_sa_institution ins
        ON aco.ins_ncode = ins.ins_ncode
    LEFT JOIN residencial res
        ON mem.per_ncode = res.res_per_ncode
    LEFT JOIN ciudad
        ON res.ciu_res_codigo = ciudad.ciu_res_codigo
    WHERE mem.per_ncode <> aco.per_ncode
)

-- ============================================================
-- PASO 5: Resultado final — titulares + beneficiarios
-- ============================================================
SELECT * FROM titular
UNION ALL
SELECT * FROM beneficiario

## Validación — Verificar el resultado

Ejecuta las siguientes celdas para confirmar que el satélite quedó cargado correctamente.

In [ ]:
%sql
-- Conteo de filas por rol
SELECT
    rol,
    COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health
GROUP BY rol
ORDER BY rol

In [ ]:
%sql
-- Total de filas del satélite
SELECT COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health

In [ ]:
%sql
-- Vista previa de los primeros 5 registros
SELECT
    mem_ncode,
    tipo_documento,
    numero_documento,
    rol,
    dir_res,
    ciudad_residencia,
    pais
FROM axa_col_slv_dv.stg_cliente.sat_beyond_health
LIMIT 5